Code for the Term Paper "Rude or Racist? Distinguishing Hate Speech from Offensive Language Using a Fine-Tuned
Transformer Model"

Zoe Alter (Matr. Nr. 108025250892)
Einführung in die Computerlinguistik
SoSe 2026
Ruhr Universität Bochum

Loading all requiered packages, datasets and models

In [ ]:
#enviroment creator

!pip3 install --upgrade pip # https://stackoverflow.com/questions/76481423/is-this-matter-error-pips-dependency-resolver-does-not-currently-take-into-ac
!pip install datasets==3.6.0

#for pandas dataframes
%pip install pandas-dedupe
%pip install "dedupe<3.0" #version incompatabilities, need this one

#to find best label for data
%pip install statistics
#to search for errors
%pip install ftfy

#for finetuning
%pip install scikit-learn #metrics
%pip install optuna


In [ ]:
#load packages
#for dataset
from datasets import load_dataset
from datasets import Dataset

#for training
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer #training
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score #evaluation
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay #visualization
import optuna #Hyperparameter search

#dataframes
import pandas as pd #for all dfs
import matplotlib.pyplot as plt #visualization
import numpy as np #visualization

#shaping data
from statistics import mode #get best label from possibles
from ftfy import fix_text #fix mujibake errors
import re #to filter mujibake
import math #for logs

#to save
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/RudeOrRacist/distilbert_hatexplain"

In [ ]:
#loading the model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3) #https://stackoverflow.com/questions/62327803/having-6-labels-instead-of-2-in-hugging-face-bertforsequenceclassification
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased") # https://towardsdatascience.com/how-to-fine-tune-distilbert-for-emotion-classification/

#reloading trained model and tokenizer
#SAVE_DIR = "/content/drive/MyDrive/RudeOrRacist/distilbert_hatexplain"

#tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
#model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
#model.eval() #in eval mode to not retrain

In [ ]:
#loading the data

#loading the dataset
dataset_id ="Hate-speech-CNERG/hatexplain"
dataset = load_dataset(dataset_id, trust_remote_code=True)  # #https://huggingface.co/docs/datasets/v2.19.0/en/package_reference/loading_methods

#inspecting data
#print(len(dataset["train"])
#print(dataset["train"])

Reshaping and preprocessing data

In [ ]:
#reshape data for readability

def reshaper(dataframe, columnname):
  """
  Takes a huggingface dataset and a column name (here subset of dataset).
  Shapes dataset into pandas dataframe and breaks open the dictionary, deleting
  unwanted columns (Id, rationales, annotators). Returns pandas dataframe with new coloumn "labels".
  """
  #reshape the set to pandas df
  df = dataframe[columnname].to_pandas()
  #reshape annotator column into label
  #explained: https://www.geeksforgeeks.org/python/apply-a-function-to-each-row-or-column-in-dataframe-using-pandas-apply/
  df["label"] = df["annotators"].apply(lambda x: x.get("label", -1))
  #drop further columns
  df = df.drop(["id", "rationales", "annotators"], axis=1)
  return df

#apply to all three subsets
df_train = reshaper(dataset, "train")
df_validation = reshaper(dataset, "validation")
df_test = reshaper(dataset, "test")

#inspect data
#df_train.head()


In [ ]:
#fixing the columns

def label_fixer(dataframe):
  """
  Takes pandas df and transforms coloumn with token lists into strings.
  Takes most common label out of label set, replacing "labels" with "true_label".
  Deletes old columns and returns edited dataframe.
  """
  #list to string
  dataframe["text"] = dataframe["post_tokens"].apply(" ".join)
  #get true label
  dataframe["true_label"] = dataframe["label"].apply(lambda x: int(mode(x))) #https://stackoverflow.com/questions/1518522/find-the-most-common-element-in-a-list
  #drop old entries
  dataframe = dataframe.drop("label", axis= 1).drop("post_tokens", axis=1)
  return dataframe

#apply to all three subsets
df_train = label_fixer(df_train)
df_validation = label_fixer(df_validation)
df_test = label_fixer(df_test)

#inspect data
#df_train.head()

In [ ]:
#Preprocessing
#searching for faulty entries

#duplicates
#https://www.geeksforgeeks.org/pandas/python-pandas-dataframe-drop_duplicates/

print(len(df_train))
print(len(df_train.drop_duplicates()))
#20 duplicates found
#drop duplicates
df_train = df_train.drop_duplicates()

#misclassifications
print(set(df_train["true_label"]))
  #no -1 labels, data properly labeled

#empty entries
print("NaN entries:\n ", df_train.isnull().sum()) #https://www.w3schools.com/python/ref_func_sum.asp
  #no NaN
#empty strings
print("empty strings: ", (df_train["text"].str.strip() == "").sum())
  #no empty strings

In [ ]:
#Preprocessing
#searching for utf errors
#https://www.nextstruggle.com/detecting-and-fixing-encoding-errors-in-legacy-files-using-python/askdushyant/

#create copy a copy and correct with fix_text
corrected_df = df_train.copy()
corrected_df["text"] = corrected_df["text"].map(lambda x: fix_text(x)) #https://medium.com/@seanmichaelhart/data-science-is-data-cleaning-4668e67deeb1


#see if any errors are present

  #convert into two lists for comparison
correct = corrected_df["text"].to_list()
base_list = df_train["text"].to_list()

  #check for differences
if correct == base_list:
  print("no mistakes!")
else: print("errors found!")
#errors were found

#create error list for manual inspection

error_dict = {}
#save every faulty entry in dict{base entry:corrected}
for i, entry in enumerate(base_list):
  if entry != correct[i]:
    error_dict[entry] = correct[i]
  else: continue

#manual inspection for error types

#print(len(error_dict)) #545 mistakes found
#correct = [x.replace("'", "’") for x in correct] #82 mistakes left
#base_list = [x.replace("‘", "’") for x in base_list] #74 mistakes left
  #most leftover errors are apostrophes. summarize this step, by filtering them out

  #filtering out apostrophe issues using regex

new_error_dict = {}
irrelevant_errors = re.compile(r"[\"“”`´'‘’]") #finds all possible aportophe errors

for i, entry in enumerate(base_list):
  if re.search(irrelevant_errors, entry): continue #ignore apostrophes
  elif entry != correct[i]: new_error_dict[entry] = correct[i] #add other errors to dict
  else: continue

print("Found this many errors: ", len(new_error_dict))
#13 entries with utf issues eliminated

#remove faulty entries from dataset
df_train_cleaned = df_train[~df_train["text"].isin(error_dict.values())]
#https://stackoverflow.com/questions/19960077/how-to-filter-pandas-dataframe-using-in-and-not-in-like-in-sql
#https://stackoverflo w.com/questions/8305199/the-tilde-operator-in-python


In [ ]:
#Preprocessing
#removing outliers
#creating data to identify outliers (TTR and text length)
#raw data heavily lobsided, searching outliers with log
#code for raw data outcommented and later on deleted

#text length
#len_list = []
len_list_log =[]

for entry in df_train_cleaned["text"]:
  #character count
  #len_list.append(len(entry))
  #reshape log
  log_length = math.log(len(entry))
  len_list_log.append(log_length)

  #safetycheck
#print(len(len_list), len_list)
#print(len(len_list_log), len_list_log)

#ttr
#ttr_list = []
ttr_list_log = []

for entry in df_train_cleaned["text"]:
  #ttr normal
  #ttr_list.append(len(set(entry.split())) / len(entry.split()))
  #ttr log
  ttr_list_log.append(math.log(len(set(entry.split())) / len(entry.split())))

  #safetycheck
#print(len(ttr_list), ttr_list)
#print(len(ttr_list_log), ttr_list_log)


#creating outlier-feature df

  #log data
outlier_data_log = {"ttr":ttr_list_log, "length":len_list_log}
log_outlier_df = pd.DataFrame(outlier_data_log, index=df_train.index) #ensure that index stays the same



In [ ]:
#Preprocessing
#removing outliers
#finding outliers using the iqr method

#log data
log_bounds= {} #entry:[low, upper]

#for ttr and text length
for name, values in log_outlier_df.items():
  #print(values)
  log_q1 = values.quantile(0.25)
  log_q3 = values.quantile(0.75)
  log_iqr = log_q3 - log_q1
  #print("quartiles", name, log_q1, log_q3, log_iqr)

  #formula for bounds
  log_lower_bound = float(log_q1 - (1.5 * log_iqr))
  log_upper_bound = float(log_q3 + (1.5 * log_iqr))
  #saving bounds
  temp = [log_lower_bound, log_upper_bound]
  log_bounds[name] = temp

#print(log_bounds)

#identify outliers in original df

 #create list of all outliers through ttr
  #https://www.tutorialspoint.com/article/how-to-search-a-value-within-a-pandas-dataframe-row
log_outlier_ttr = log_outlier_df.loc[
    (log_outlier_df['ttr'] < log_bounds['ttr'][0]) | #from outlier df, add to list all outliers that are ABOVE
    (log_outlier_df['ttr'] > log_bounds['ttr'][1])   #or BELOW the relevant bounds
]
print("TTR based (log): ", len(log_outlier_ttr))#, log_outlier_ttr)

#create list of all outliers through textlength
log_outlier_len = log_outlier_df.loc[
    (log_outlier_df['length'] < log_bounds['length'][0]) | #from outlier df, add to list all outliers that are ABOVE
    (log_outlier_df['length'] > log_bounds['length'][1])   #or BELOW the relevant bounds
]
print("Length based (log)", len(log_outlier_len)) #log_outlier_len)

#use index to find outliers in base df
  #save in a list for inspection
log_ttr_outliers = []
log_len_outliers = []

for outlier in log_outlier_ttr.index:
  log_ttr_outliers.append(df_train.loc[outlier])

for outlier in log_outlier_len.index:
  log_len_outliers.append(df_train.loc[outlier])

#print(len(log_ttr_outliers), log_ttr_outliers)
#print(len(log_len_outliers), log_len_outliers)


In [ ]:
#Preprocessing
#eliminating outliers

#LOG
#https://www.w3schools.com/python/ref_set_union.asp
#https://medium.com/@heyamit10/understanding-how-to-drop-an-index-in-pandas-6c011d0d3764
df_log_train = df_train.drop(index=log_outlier_ttr.index.union(log_outlier_len.index))

print(len(df_train), len(df_log_train))
print(df_log_train.head())

In [ ]:
#Compare label distribution
#before and after preprocessing

#https://www.w3schools.com/python/matplotlib_pie_charts.

#preclean
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
labels = ["hatespeech", "neither", "offensive"]

# preclean
y_preclean = np.array([
    (df_train["true_label"] == 0).sum(), #hatespeech
    (df_train["true_label"] == 1).sum(), #normal
    (df_train["true_label"] == 2).sum() #offensive
])

axes[0].pie(y_preclean, labels=labels, autopct='%.2f%%')
axes[0].set_title("Preclean")

# after outlier removal
y_log = np.array([
    (df_log_train["true_label"] == 0).sum(), #hatespeech
    (df_log_train["true_label"] == 1).sum(), #normal
    (df_log_train["true_label"] == 2).sum() #offensive
])

axes[1].pie(y_log, labels=labels, autopct='%.2f%%')
axes[1].set_title("Log Outlier Removal")

fig.legend(labels, title="Categories:", loc="lower center", ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
#Convert DF to Destilbert dict
#https://towardsdatascience.com/how-to-fine-tune-distilbert-for-emotion-classification/

#function
def preprocess_function(BATCH):
    """
    Tokenizes text data and transforms labels into numerical IDs.
    Takes a df with columns "text" and "true_label".

    Returns a dictionary encoded for Destilbert with:
      input id: encoded token sequence
      attention_mask: mask to indicate padding tokens
      label: numerical labels for classification

    Example usage:
        train_dataset = train_dataset.map(lambda x: preprocess_function(x, tokenizer), batched=True)
    """

    tokenized_inputs = tokenizer(
        BATCH["text"],
        padding="max_length", #padding for max length
        truncation=True, #truncate after max length
        max_length=512, #set max length in symbols
        return_tensors="pt" #return tensors
    )

    tokenized_inputs["label"] = BATCH["true_label"]
    return tokenized_inputs



#application

train_log_dataset = Dataset.from_pandas(df_log_train) #Best results in raw test run, remains base
#train_raw_dataset = Dataset.from_pandas(df_train)
train_dataset = train_log_dataset.map(preprocess_function, batched=True) #https://www.geeksforgeeks.org/python/python-map-function/

#repeat for validation
validation_dataset = Dataset.from_pandas(df_validation)
validation_dataset = validation_dataset.map(preprocess_function, batched=True)

#repeat for test
test_dataset = Dataset.from_pandas(df_test)
test_dataset = test_dataset.map(preprocess_function, batched=True)


Preparing the model

In [ ]:
#define metrics for model evaluation
#https://towardsdatascience.com/how-to-fine-tune-distilbert-for-emotion-classification/

def compute_metrics(p):
    """
    Computes accuracy, F1 score, precision, and recall metrics for multiclass classification.
    Takes p, a tuple containing logits and labels.
    returns dict: Dictionary with accuracy, F1 score, precision, and recall metrics,
    using weighted averaging to account for class imbalance in multiclass classification tasks.
    """
    logits, labels = p

    # Convert logits to probabilities using softmax (PyTorch)
    softmax = torch.nn.Softmax(dim=1)
    probs = softmax(torch.tensor(logits))

    # Convert logits to predicted class labels
    preds = probs.argmax(axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),  # Accuracy metric
        "f1_score": f1_score(labels, preds, average='weighted'),  # F1 score with weighted average for imbalanced data
        "precision": precision_score(labels, preds, average='weighted'),  # Precision score with weighted average
        "recall": recall_score(labels, preds, average='weighted'),  # Recall score with weighted average
    }

In [ ]:
#overfitting on sampleset as sanity check
  #overfitting as proof of good starting model: https://stackoverflow.com/questions/45342766/how-can-i-intentionally-overfit-a-convolutional-neural-net-in-keras-to-make-sure
  #successfull, reached training loss = 0.006563


#create temporary set (for overfitting with 32 entries, for optuna with 1000)
pre_test_df = df_log_train.groupby("true_label").sample(1000, random_state=222) # Source - https://stackoverflow.com/a/66903676
pre_test_dataset = Dataset.from_pandas(pre_test_df)
pre_test_dataset = pre_test_dataset.map(preprocess_function, batched=True)

#finetune on sample set
#most arguments cut, since we only care for training loss in this step
#https://towardsdatascience.com/how-to-fine-tune-distilbert-for-emotion-classification/

#hyperparameters
#standard with high epochs for overfitting
lr = 2e-5
batch_size = 16
num_epochs = 50
weight_decay = 0.01

#set up training arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=weight_decay,
    logging_steps=20, #log progress every 20 steps
)

 #initialize trainer, with our model, arguments and dataset
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=pre_test_dataset,
    eval_dataset=pre_test_dataset,
    compute_metrics=compute_metrics,
)

# Train the model on sampleset
#trainer.train()

In [ ]:
#freezing first layers of model
#prevents overfitting, saves resources
#https://towardsdatascience.com/how-to-fine-tune-distilbert-for-emotion-classification/

# freeze base model parameters
for name, param in model.base_model.named_parameters(): #https://huggingface.co/transformers/v3.0.2/model_doc/auto.html
    param.requires_grad = False

# keep classifier trainable, by keeping ultimate layer (layer 6, index 5) adjustable
    if "transformer.layer.5" in name or "classifier" in name:
        param.requires_grad = True

In [ ]:
#Identifying ideal hyperparameters with optuna
#https://huggingface.co/docs/transformers/hpo_train


  #ranges from:
  #https://massedcompute.com/faq-answers/?question=What+is+the+optimal+value+for+weight+decay+in+BERT+model+training?
  #https://medium.com/we-talk-data/deep-learning-basics-practical-guide-to-weight-decay-in-pytorch-d9e26fc669db
  #https://arxiv.org/abs/1810.04805


#load model and freezing each trial (def)
def model_init(trial):
  """
  load and freeze model
  returns model
  """
    #load
  model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)
    #freeze, same as before
  for name, param in model.base_model.named_parameters():
    param.requires_grad = False
  for name, param in model.named_parameters():
    if "transformer.layer.5" in name or "classifier" in name:
      param.requires_grad = True
    #return loaded model
  return model


#define f1 as target metric (def)
def compute_objective(metrics):
  """
  takes metrics, returns f1 score
  """
  return metrics["eval_f1_score"]


# define searchspace for hyperparameters, testing each trial (def)
def hp_space(trial):
  """
  suggest values from search space:
  learning rate (float), weight decay (float), batch size (16 or 32), epoch (int)
  returns dict: Dictionary with suggested hyperparameters.
  """
  return {
        "learning_rate": trial.suggest_float("learning_rate", 2e-5, 1e-4, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-4, 0.1, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [16, 32]),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 4),  # NEU
    }


#set up trainings arguments
  #even shorter than overfitting test
search_args = TrainingArguments(
    output_dir="./hpo_results",
    eval_strategy="epoch",
    save_strategy="no",
    fp16=True,
    report_to="none",
)


#set up trainer
trainer = Trainer(
    model_init=model_init,
    args=search_args,
    train_dataset=pre_test_dataset,
    eval_dataset=validation_dataset,
    compute_metrics=compute_metrics,
)


#Search:
best_run = trainer.hyperparameter_search(
    direction="maximize", #find best value
    backend="optuna", #using optuna
    hp_space=hp_space, #in the defined space
    compute_objective=compute_objective #for f1
)
print(best_run.hyperparameters)
print(best_run.objective)

Best parameters found with optuna:

lr = 8.406899834793315e-05
batch_size = 16
num_epochs = 4
weight_decay = 0.0014726284160216136


In [ ]:
#Finetuning the model
#https://towardsdatascience.com/how-to-fine-tune-distilbert-for-emotion-classification/
#https://discuss.huggingface.co/t/logging-experiment-tracking-with-w-b/498


# Define hyperparameters
  #taken from optuna results
lr = 8.406899834793315e-05
batch_size = 16
num_epochs = 4
weight_decay = 0.0014726284160216136


# Set up training arguments for fine-tuning models
training_args = TrainingArguments(
    #optim="adamw_torch" #standard
    output_dir="/content/drive/MyDrive/RudeOrRacist/results",
    eval_strategy="steps", #evaluate in steps
    eval_steps=250, #evaluate every 250 steps
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=weight_decay,
    logging_steps=250, #log every 250 steps
    load_best_model_at_end=True, #save best model from finetuning (f1), not final
    metric_for_best_model="eval_f1_score", #evaluate using f1 score
    greater_is_better=True, #evaluate using max f1 score
    report_to="wandb" #visualization
)


# Initialize the Trainer with the model, arguments, and datasets
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

In [ ]:
#Evaluate the model (best f1)
print(trainer.evaluate())

In [ ]:
#evaluate model on test set
final_results = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
#https://huggingface.co/docs/transformers/main_classes/trainer

In [ ]:
#Visualize results

#confusion matrix
#https://www.w3schools.com/python/python_ml_confusion_matrix.asp
pred = trainer.predict(test_dataset, metric_key_prefix="test")

y_pred = pred.predictions.argmax(axis=1)
y_true = pred.label_ids

cm = confusion_matrix(y_true, y_pred)

cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels = [0,1,2])

cm_display.plot(cmap=plt.cm.Blues)
#disp.plot(cmap=plt.cm.Blues) https://stackoverflow.com/questions/68914814/sklearn-confusion-matrix-how-to-get-the-blue-color/68919012
plt.show()


In [ ]:
# Visualize observations that were incorrectly labeled
predictions = list(y_pred)
true_labels = list(y_true)

#get dict with false prediction entries
false_predictions = {}

for index, entry in enumerate(predictions):
  if entry != true_labels[index]:

    temp_key = (int(true_labels[index]), int(entry))

    if temp_key not in false_predictions:
      false_predictions[temp_key] = [test_dataset["text"][index],]
    else:
      false_predictions[temp_key].append(test_dataset["text"][index])


#get overview of entries
for combi, lists in false_predictions.items():
  print(combi,len(lists))
  for entry in lists:
    print(entry)

In [ ]:
#save the model
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)